<div class="blog-language-switch" role="group" aria-label="Article language">
<span aria-current="page">English</span>
<a href="../zh-CN/Machine-Learning/09-rules-trees-ensembles.html" lang="zh-CN" hreflang="zh-CN">中文</a>
</div>

[Back to Machine Learning guideline](Machine Learning.html)


## **Rule Learning, Decision Trees, and Ensembles**

Rule learners and decision trees describe a prediction as a collection of conditional decisions. A linear model asks whether one weighted sum crosses a boundary; a tree asks a sequence of questions such as `age <= 35?` and `income > 70,000?`. Each answer routes the observation toward a smaller region of the input space, where a simple class probability or numerical value is returned. This makes trees naturally nonlinear, able to represent interactions, and largely insensitive to feature scaling.

A single deep tree is flexible but unstable: a small change in the training sample can change an early split and therefore rebuild much of the tree. Ensemble methods turn that weakness into an advantage. **Bagging** and **random forests** average many deliberately varied trees to reduce variance. **Boosting** builds trees sequentially so that each stage corrects weaknesses of the current ensemble. **Voting** and **stacking** combine models that may come from different families.

The progression is therefore not simply "one model after another." It is a sequence of answers to three questions:

1. How can conditions be converted into explicit prediction rules?
2. How can a tree search for and organize those conditions automatically?
3. How can multiple imperfect trees be combined without leaking validation information?

<div class="diagram-scroll">

![A map from explicit rules and a single decision tree to bagging, boosting, voting, and stacking.](assets/tree-ensemble-learning-map.svg){fig-alt="A map from explicit rules and a single decision tree to bagging, boosting, voting, and stacking."}

</div>

This chapter focuses on supervised tabular learning. Trees also appear inside anomaly detection, survival analysis, causal forests, ranking systems, and reinforcement learning, but those extensions reuse the same ideas of recursive partitioning, regularization, randomization, and aggregation developed here.


### **Rule-Based Learning**

A classification rule has an **antecedent**, a conjunction of feature conditions, and a **consequent**, the prediction made when those conditions are satisfied:

$$
\underbrace{(x_{j_1}\in A_1)\land\cdots\land(x_{j_m}\in A_m)}_{\text{antecedent}}
\quad\Longrightarrow\quad
\underbrace{\hat y=c}_{\text{consequent}}.
$$

The antecedent defines a region of the input space. A case is **covered** when it lies in that region. Rule systems are attractive when a decision must be inspected, challenged, or converted into an operational policy. They can express threshold logic, exceptions, and interactions without asking the reader to interpret coefficients on transformed features.

Interpretability does not make a rule automatically reliable. A very specific rule can achieve perfect training precision by covering only one observation. A useful rule must therefore be evaluated by both **purity** and **coverage**, and the complete rule set needs an explicit policy for conflicts, uncovered cases, and missing values.

<div class="diagram-scroll">

![OneR, separate-and-conquer, and decision-tree strategies for learning prediction rules.](assets/rule-learning-strategies.svg){fig-alt="OneR uses one attribute, separate-and-conquer grows one rule at a time, and a decision tree recursively partitions all observations."}

</div>

#### **One Rule (OneR)**

OneR is a deliberately simple classification baseline. For each candidate feature, it assigns every observed feature value to that value's majority class. It then selects the feature whose resulting rule table makes the fewest training errors. With feature $j$, value $v$, and training index set $I_{jv}=\{i:x_{ij}=v\}$, the value-specific prediction is

$$
\hat c_{jv}=\arg\max_c\sum_{i\in I_{jv}}\mathbf 1(y_i=c),
$$

and the feature's empirical error is

$$
\operatorname{Err}(j)=\sum_v\sum_{i\in I_{jv}}\mathbf 1(y_i\ne \hat c_{jv}).
$$

OneR is useful because it answers a sharp baseline question: **how much can be predicted from the single best attribute?** If a sophisticated pipeline barely beats it, the complex model may not be earning its operational cost. The method can also expose leakage, such as an identifier-like field that almost determines the target.

For a continuous feature, the raw values must first be discretized or searched for useful thresholds. That transformation must be fitted inside each training fold. High-cardinality categorical features are another hazard because small value groups can memorize labels. Unknown values at inference require a default class or an explicit `unknown` bucket.

```text
for each feature j:
    for each observed value v of feature j:
        predict the majority class among rows with x_j = v
    count the errors made by this feature's rule table
return the feature and rule table with the lowest error
```

<details>
<summary><strong>Python example: implement OneR for categorical features</strong></summary>

```python
from collections import Counter
import numpy as np

class OneR:
    def fit(self, X, y, feature_names=None):
        X = np.asarray(X, dtype=object)
        y = np.asarray(y, dtype=object)
        self.default_class_ = Counter(y).most_common(1)[0][0]
        self.feature_names_ = feature_names or [f"x{j}" for j in range(X.shape[1])]

        best = None
        for j in range(X.shape[1]):
            rules = {}
            errors = 0

            # Each observed value receives its local majority class.
            for value in np.unique(X[:, j]):
                mask = X[:, j] == value
                majority = Counter(y[mask]).most_common(1)[0][0]
                rules[value] = majority
                errors += np.sum(y[mask] != majority)

            candidate = (errors, j, rules)
            if best is None or candidate[0] < best[0]:
                best = candidate

        self.training_errors_, self.feature_index_, self.rules_ = best
        return self

    def predict(self, X):
        X = np.asarray(X, dtype=object)
        return np.array([
            self.rules_.get(value, self.default_class_)
            for value in X[:, self.feature_index_]
        ])

X = np.array([
    ["sunny", "high", "weak"],
    ["sunny", "high", "strong"],
    ["overcast", "high", "weak"],
    ["rain", "high", "weak"],
    ["rain", "normal", "weak"],
    ["rain", "normal", "strong"],
    ["overcast", "normal", "strong"],
    ["sunny", "normal", "weak"],
], dtype=object)
y = np.array(["no", "no", "yes", "yes", "yes", "no", "yes", "yes"])

model = OneR().fit(X, y, feature_names=["outlook", "humidity", "wind"])
print("selected feature:", model.feature_names_[model.feature_index_])
print("rules:", model.rules_)
print("training errors:", model.training_errors_)
print("unknown-value prediction:", model.predict([["snow", "high", "weak"]])[0])
```

</details>

#### **Separate-and-Conquer and PRISM**

OneR cannot represent interactions. **Separate-and-conquer** algorithms instead construct one rule at a time. PRISM is a classic example. To learn a rule for target class $c$, it starts with all rows and repeatedly adds the condition with the largest conditional class precision

$$
\operatorname{precision}(A=v\mid R,c)=\frac{p}{t},
$$

where $t$ is the number of currently covered rows that also satisfy $A=v$, and $p$ is the number of those rows belonging to $c$. Conditions are added until the rule covers no negative examples or no useful refinement remains. Covered positive rows are then removed, and another rule is grown for the same class.

The name describes the search strategy:

- **Separate:** identify a high-purity region for one class.
- **Conquer:** record a rule for that region and remove its covered positives.
- **Repeat:** continue until the target cases are covered, then move to another class.

```text
for each target class c:
    while uncovered examples of c remain:
        start an empty rule R
        while R still covers examples from other classes:
            evaluate every unused attribute-value condition
            add the condition maximizing target precision p / t
            break ties using larger positive coverage p
        store R -> c
        remove positive examples covered by R
```

Maximizing raw $p/t$ is greedy and can prefer a condition supported by a single positive row. Minimum coverage, Laplace correction, validation-based pruning, or a complexity penalty is therefore needed in noisy data. PRISM is best understood as an interpretable search procedure, not as a guarantee that the discovered rules are causal or stable.

<details>
<summary><strong>Python example: grow PRISM-style rules for one target class</strong></summary>

```python
import numpy as np

def prism_rules_for_class(X, y, feature_names, target):
    X = np.asarray(X, dtype=object)
    y = np.asarray(y, dtype=object)
    remaining_positive = set(np.flatnonzero(y == target))
    active = np.ones(len(y), dtype=bool)
    learned = []

    while remaining_positive:
        # Previously covered positives are removed from later rule searches.
        covered = active.copy()
        conditions = []
        unused = set(range(X.shape[1]))

        while np.any(covered & (y != target)) and unused:
            best = None
            for j in unused:
                for value in np.unique(X[covered, j]):
                    candidate_mask = covered & (X[:, j] == value)
                    total = int(candidate_mask.sum())
                    positives = int(np.sum(candidate_mask & (y == target)))
                    if positives == 0:
                        continue
                    score = positives / total
                    # Precision first, then positive coverage, then deterministic tie-breaks.
                    key = (score, positives, -j, str(value))
                    if best is None or key > best[0]:
                        best = (key, j, value, candidate_mask)

            if best is None:
                break
            _, j, value, covered = best
            conditions.append((feature_names[j], value))
            unused.remove(j)

        covered_positive = set(map(int, np.flatnonzero(covered & (y == target))))
        if not covered_positive:
            break
        learned.append({"conditions": conditions, "predict": target,
                        "covered_positive": sorted(covered_positive)})
        remaining_positive -= covered_positive
        active[list(covered_positive)] = False

    return learned

X = [
    ["sunny", "high", "weak"], ["sunny", "high", "strong"],
    ["overcast", "high", "weak"], ["rain", "high", "weak"],
    ["rain", "normal", "weak"], ["rain", "normal", "strong"],
    ["overcast", "normal", "strong"], ["sunny", "normal", "weak"],
]
y = ["no", "no", "yes", "yes", "yes", "no", "yes", "yes"]

rules = prism_rules_for_class(X, y, ["outlook", "humidity", "wind"], "yes")
for number, rule in enumerate(rules, start=1):
    print(f"rule {number}: IF {rule['conditions']} THEN yes; rows={rule['covered_positive']}")
```

</details>

#### **Rule Quality, Ordering, and Defaults**

Suppose rule $R$ predicts class $c$. Let $C_R$ be covered rows and $P_c$ all rows of class $c$. Several quantities answer different questions:

$$
\begin{aligned}
\operatorname{coverage}(R)&=\frac{|C_R|}{n},\\
\operatorname{precision}(R\to c)&=\frac{|C_R\cap P_c|}{|C_R|},\\
\operatorname{recall}(R\to c)&=\frac{|C_R\cap P_c|}{|P_c|},\\
\operatorname{lift}(R\to c)&=\frac{\operatorname{precision}(R\to c)}{|P_c|/n}.
\end{aligned}
$$

Coverage describes reach, precision describes reliability within the rule, recall describes how much of the target class it captures, and lift compares its precision with the base rate. For a binary rule with $p$ covered positives and $q$ covered negatives, Laplace-corrected precision

$$
\frac{p+1}{p+q+2}
$$

pulls tiny perfect rules toward $1/2$, making unsupported certainty less attractive.

An **ordered rule list** applies the first matching rule, so precedence is part of the model. An **unordered rule set** needs a conflict strategy such as confidence-weighted voting. Both require a default for uncovered cases. These details should be evaluated as part of the pipeline because changing rule order can change predictions without changing any individual rule.

<details>
<summary><strong>Python example: calculate rule coverage, precision, recall, lift, and Laplace score</strong></summary>

```python
import numpy as np

# A rule predicts churn when usage is low and support calls are high.
usage = np.array([2, 8, 3, 7, 1, 4, 9, 2, 6, 3])
calls = np.array([5, 1, 4, 2, 6, 3, 0, 5, 1, 4])
churn = np.array([1, 0, 1, 0, 1, 0, 0, 1, 0, 0], dtype=bool)
covered = (usage <= 3) & (calls >= 4)

p = int(np.sum(covered & churn))
q = int(np.sum(covered & ~churn))
coverage = covered.mean()
precision = p / (p + q)
recall = p / churn.sum()
base_rate = churn.mean()
lift = precision / base_rate
laplace = (p + 1) / (p + q + 2)

print(f"covered positives={p}, covered negatives={q}")
print(f"coverage={coverage:.2f}, precision={precision:.2f}, recall={recall:.2f}")
print(f"lift={lift:.2f}, Laplace precision={laplace:.2f}")
```

</details>

| Model | Search unit | Main strength | Main limitation |
|---|---|---|---|
| OneR | One feature and its value table | Extremely transparent baseline | Cannot model feature interactions |
| PRISM / covering | One conjunction for one class at a time | Direct class-specific rules | Greedy rules can overlap and overfit |
| Decision tree | A shared hierarchy of recursive splits | Compact reuse of earlier conditions | Greedy topology is unstable |
| Rule ensemble | Many rules with learned weights | Flexible while retaining rule features | Less immediately readable than a short list |


### **Decision Tree Foundations**

A decision tree is a hierarchical rule system. Each internal node asks a question about one feature, each outgoing edge represents an answer, and each leaf stores a prediction. A root-to-leaf path is therefore an ordered conjunction of conditions. For ordinary numerical trees, conditions are usually axis-aligned thresholds such as $x_j\le s$; for categorical variables they may test membership in a category subset.

Trees solve several problems that are awkward for a single linear boundary:

- **Nonlinearity:** different regions can receive unrelated predictions.
- **Interactions:** a feature can matter only after an earlier condition is satisfied.
- **Mixed scales:** split ordering is unchanged by strictly monotone rescaling, so standardization is usually unnecessary.
- **Local explanations:** a prediction can be traced through the conditions that produced it.

Their main limitation follows from the same flexibility. Greedy splitting can fit accidental local patterns, and axis-aligned partitions may need many leaves to approximate a diagonal or smooth boundary.

#### **Recursive Partitioning**

At node $t$, let $I_t$ be the indices reaching that node and let $Q(I_t)$ be its impurity. A candidate split $(j,s)$ creates

$$
I_L(j,s)=\{i\in I_t:x_{ij}\le s\},\qquad
I_R(j,s)=I_t\setminus I_L(j,s).
$$

The standard greedy tree chooses the split minimizing weighted child impurity,

$$
(j^*,s^*)=\arg\min_{j,s}
\left[
\frac{|I_L|}{|I_t|}Q(I_L)+
\frac{|I_R|}{|I_t|}Q(I_R)
\right].
$$

Equivalently, it maximizes the impurity reduction from the parent. The process repeats independently in each child until a stopping rule is met. Because the algorithm commits to the best **current** split, it does not guarantee the globally smallest or most accurate tree. Finding an optimal tree under common size constraints is computationally hard, which is why practical implementations use greedy induction followed by regularization or pruning.

```text
grow(node data I):
    if a stopping condition is satisfied:
        return a leaf prediction estimated from I
    evaluate candidate feature/threshold splits
    choose the split with the largest impurity decrease
    return node(split, grow(left subset), grow(right subset))
```

The following official example makes the geometry visible. Each split adds a vertical or horizontal boundary; their intersections form rectangular prediction regions. Some feature pairs separate the Iris classes easily, while others require narrow, fragile regions.

![Decision surfaces of decision trees trained on pairs of Iris features.](assets/tree-decision-surfaces.png){fig-alt="Six axis-aligned decision surfaces learned by trees from pairs of Iris features."}

*Official scikit-learn illustration: [decision-tree surfaces on Iris](https://scikit-learn.org/stable/auto_examples/tree/plot_iris_dtc.html).*

The tree itself stores the same partition as a hierarchy. A classification leaf estimates class probabilities from the (possibly weighted) class proportions among its training cases; prediction returns the largest probability. A regression leaf usually returns a mean, median, or another loss-optimal constant.

![A decision tree trained on the Iris features, with threshold, impurity, sample count, and class composition at each node.](assets/tree-structure-iris.png){fig-alt="A plotted Iris decision tree showing thresholds, Gini impurity, sample counts, and class distributions."}

*Official scikit-learn illustration: [tree structure on Iris](https://scikit-learn.org/stable/auto_examples/tree/plot_iris_dtc.html).*

<details>
<summary><strong>Python example: fit, print, and trace a classification tree</strong></summary>

```python
import numpy as np
from sklearn.datasets import load_iris
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score
from sklearn.tree import DecisionTreeClassifier, export_text

iris = load_iris()
X_train, X_test, y_train, y_test = train_test_split(
    iris.data, iris.target, test_size=0.30, stratify=iris.target, random_state=7
)

tree = DecisionTreeClassifier(
    max_depth=3, min_samples_leaf=4, criterion="gini", random_state=7
)
tree.fit(X_train, y_train)
prediction = tree.predict(X_test)

print("test accuracy:", round(accuracy_score(y_test, prediction), 3))
print("tree depth:", tree.get_depth(), "leaves:", tree.get_n_leaves())
print(export_text(tree, feature_names=list(iris.feature_names)))

# decision_path returns the internal node IDs visited by this case.
case = X_test[[0]]
visited = tree.decision_path(case).indices
leaf = tree.apply(case)[0]
probabilities = tree.predict_proba(case)[0]
print("visited node IDs:", visited.tolist())
print("terminal leaf:", int(leaf))
print("class probabilities:", np.round(probabilities, 3))
```

</details>

#### **Entropy and Information Gain**

For a node containing class proportions $p_1,\ldots,p_K$, Shannon entropy is

$$
H(t)=-\sum_{k=1}^{K}p_k\log_2p_k,
$$

with the convention $0\log 0=0$. Entropy is zero when one class occupies the entire node and is largest for a uniform class distribution. In binary classification, its maximum is one bit at $p_1=p_2=1/2$.

The information gain of a split is the reduction in uncertainty:

$$
\operatorname{IG}=H(t)-
\frac{n_L}{n_t}H(t_L)-
\frac{n_R}{n_t}H(t_R).
$$

A split that creates pure children has high gain. The child entropies must be weighted by child size: isolating one pure observation is less valuable than separating a large coherent group. Entropy can be interpreted through coding length, but in tree induction it is principally a local class-purity criterion, not proof that a split contains causal information.

#### **Gini Impurity**

Gini impurity is

$$
G(t)=1-\sum_{k=1}^{K}p_k^2=\sum_{k=1}^{K}p_k(1-p_k).
$$

It equals the probability that two labels independently drawn from the node's class distribution differ. For two classes, $G(t)=2p(1-p)$ and reaches $1/2$ at a balanced node. Gini avoids logarithms and is the common CART classification criterion. Entropy and Gini usually rank strong splits similarly, but not identically; the choice is normally less important than depth, leaf size, and validation design.

<details>
<summary><strong>Python example: search numerical thresholds with entropy and Gini impurity</strong></summary>

```python
import numpy as np

x = np.array([1.0, 1.8, 2.4, 3.1, 4.0, 4.7, 5.3, 6.0])
y = np.array([0,   0,   0,   1,   0,   1,   1,   1])

def entropy(labels):
    counts = np.bincount(labels, minlength=2)
    p = counts[counts > 0] / len(labels)
    return -np.sum(p * np.log2(p))

def gini(labels):
    p = np.bincount(labels, minlength=2) / len(labels)
    return 1.0 - np.sum(p ** 2)

def best_threshold(impurity):
    # Only midpoints between consecutive unique values can change a split.
    candidates = (x[:-1] + x[1:]) / 2
    parent = impurity(y)
    scored = []
    for threshold in candidates:
        left = y[x <= threshold]
        right = y[x > threshold]
        weighted = (len(left) * impurity(left) + len(right) * impurity(right)) / len(y)
        scored.append((parent - weighted, threshold, len(left), len(right)))
    return max(scored)

for name, criterion in [("entropy", entropy), ("gini", gini)]:
    gain, threshold, n_left, n_right = best_threshold(criterion)
    print(f"{name:>7}: threshold={threshold:.2f}, reduction={gain:.3f}, "
          f"sizes=({n_left}, {n_right})")
```

</details>

#### **Regression Tree Criteria**

A regression tree replaces class impurity with within-node prediction loss. Under squared error, the optimal leaf value is the node mean $\bar y_t$, and impurity is

$$
Q_{\text{SE}}(t)=\frac{1}{n_t}\sum_{i\in I_t}(y_i-\bar y_t)^2.
$$

Minimizing weighted child squared error chooses splits that create groups with internally similar targets. Absolute-error trees use a median and are more resistant to extreme targets but can be slower to fit. Poisson deviance is useful for nonnegative count-like targets when its distributional assumptions are reasonable.

Because every leaf returns one constant, a regression tree is a **piecewise-constant approximator**. Increasing depth creates finer steps: it can capture more local structure, but eventually follows noise and extrapolates poorly beyond the observed feature range.

![Regression trees of different maximum depths fitted to noisy one-dimensional data.](assets/tree-regression-depth.png){fig-alt="Noisy one-dimensional regression data with shallow and deeper stepwise decision-tree fits."}

*Official scikit-learn illustration: [decision-tree regression and depth](https://scikit-learn.org/stable/auto_examples/tree/plot_tree_regression.html).*

<details>
<summary><strong>Python example: observe depth, overfitting, and stepwise regression predictions</strong></summary>

```python
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.tree import DecisionTreeRegressor
from sklearn.metrics import root_mean_squared_error

rng = np.random.default_rng(12)
X = np.linspace(0, 6, 180).reshape(-1, 1)
signal = np.sin(X[:, 0]) + 0.25 * X[:, 0]
y = signal + rng.normal(0, 0.22, size=len(X))
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.35, random_state=12
)

for depth in [2, 5, None]:
    model = DecisionTreeRegressor(max_depth=depth, min_samples_leaf=1, random_state=12)
    model.fit(X_train, y_train)
    train_rmse = root_mean_squared_error(y_train, model.predict(X_train))
    test_prediction = model.predict(X_test)
    test_rmse = root_mean_squared_error(y_test, test_prediction)
    unique_steps = len(np.unique(np.round(test_prediction, 8)))
    print(f"depth={str(depth):>4}: leaves={model.get_n_leaves():3d}, "
          f"train RMSE={train_rmse:.3f}, test RMSE={test_rmse:.3f}, "
          f"test step values={unique_steps}")
```

</details>

Classification and regression trees share the same recursive machinery. The target type changes the impurity and leaf estimate, while pruning and ensembling address the same instability in both settings.


### **Tree Induction Families and Split Handling**

"Decision tree" names a model family, not one unique algorithm. The historical algorithms differ in their split criterion, whether a node can have more than two children, how continuous and missing values are handled, and how the final tree is pruned.

#### **ID3, C4.5, and CART**

| Family | Typical target | Split form | Criterion | Pruning and notable behavior |
|---|---|---|---|---|
| ID3 | Classification | Originally multiway categorical splits | Information gain | Original form has limited continuous/missing handling and no principled post-pruning |
| C4.5 | Classification | Multiway categorical or threshold splits | Gain ratio | Error-based post-pruning; improved continuous and missing-value handling |
| CART | Classification and regression | Strictly binary splits | Gini for classification; squared/absolute/Poisson-style losses for regression | Minimal cost-complexity pruning; surrogate splits in the classical algorithm |

Modern libraries borrow ideas across these families. For example, a binary tree can use entropy, and a gradient-boosted tree can use histogram-based threshold search. It is more precise to inspect an implementation's objective and missing-value behavior than to infer everything from a historical family name.

Information gain favors attributes with many possible outcomes. A unique identifier can create pure one-row children and obtain maximal gain even though it cannot generalize. C4.5's **gain ratio** divides information gain by the split's own entropy,

$$
\operatorname{GainRatio}(A)=
\frac{\operatorname{IG}(A)}{\operatorname{SplitInfo}(A)},
\qquad
\operatorname{SplitInfo}(A)=-\sum_v\frac{n_v}{n}\log_2\frac{n_v}{n}.
$$

This penalizes highly fragmented splits, although it can behave erratically when `SplitInfo` is very small. C4.5 therefore considers gain ratio together with a minimum information-gain requirement rather than treating the ratio as an unrestricted objective.

<details>
<summary><strong>Python example: see information gain prefer an identifier and gain ratio penalize it</strong></summary>

```python
import numpy as np

y = np.array([0, 0, 0, 0, 0, 1, 0, 1, 1, 1, 1, 1])
identifier = np.array([f"id-{i}" for i in range(len(y))])
useful_signal = np.array(["A"] * 6 + ["B"] * 6)

def entropy(labels):
    _, counts = np.unique(labels, return_counts=True)
    p = counts / counts.sum()
    return -np.sum(p * np.log2(p))

def categorical_gain_and_ratio(feature, target):
    parent = entropy(target)
    weighted_child_entropy = 0.0
    split_info = 0.0
    for value in np.unique(feature):
        mask = feature == value
        weight = mask.mean()
        weighted_child_entropy += weight * entropy(target[mask])
        split_info -= weight * np.log2(weight)
    gain = parent - weighted_child_entropy
    ratio = gain / split_info if split_info > 0 else 0.0
    return gain, ratio, len(np.unique(feature))

for name, feature in [("identifier", identifier), ("useful signal", useful_signal)]:
    gain, ratio, branches = categorical_gain_and_ratio(feature, y)
    print(f"{name:>13}: branches={branches:2d}, information gain={gain:.3f}, "
          f"gain ratio={ratio:.3f}")
```

</details>

#### **Categorical and Continuous Splits**

For a sorted numerical feature with unique values $v_1<\cdots<v_m$, a binary tree only needs to examine thresholds between adjacent values whose routing can differ, commonly

$$
s_r=\frac{v_r+v_{r+1}}{2},\qquad r=1,\ldots,m-1.
$$

Efficient implementations sort features once or use histograms rather than repeatedly scanning every raw value. Exact complexity depends on data layout, sparsity, and implementation, but split search rather than final prediction is usually the expensive part of fitting a single tree.

A categorical feature with $K$ levels is harder. An unconstrained binary split can choose a subset $S$ and ask $x_j\in S$, creating $2^{K-1}-1$ distinct nontrivial partitions. Exhaustive search quickly becomes infeasible. Common strategies include ordering categories by target statistics, one-hot encoding, native partition search, or ordered target encoding. Integer-label encoding alone silently imposes an arbitrary order and should not be treated as a harmless representation.

Target encoding must be cross-fitted. If each category's encoding is calculated using the same row's target, rare levels leak label information before the tree is fitted. Chapter 02's preprocessing rules still apply even though tree models do not need feature standardization.

#### **Missing Values and Surrogate Splits**

Missingness can carry information, but it first needs a defined routing policy. Common approaches are:

- impute a value learned from the training fold, optionally adding a missingness indicator;
- learn a default direction for missing cases at each split;
- treat missingness as a separate histogram bin;
- use a **surrogate split**, another feature whose partition most closely reproduces the primary split.

A surrogate is not simply the next most predictive feature. It is selected by agreement with the primary split. If `income <= 60k` is the preferred question but income is absent, a correlated feature such as job grade may approximate the same left/right routing. Classical CART stores a ranked list of these alternatives. Native missing-value behavior varies by library and estimator, so it must be checked rather than assumed.

<details>
<summary><strong>Python example: compare imputation with a tree model that routes NaN values natively</strong></summary>

```python
import numpy as np
from sklearn.datasets import make_classification
from sklearn.ensemble import HistGradientBoostingClassifier
from sklearn.impute import SimpleImputer
from sklearn.metrics import roc_auc_score
from sklearn.model_selection import train_test_split
from sklearn.pipeline import make_pipeline
from sklearn.tree import DecisionTreeClassifier

X, y = make_classification(
    n_samples=1400, n_features=8, n_informative=5, class_sep=0.9, random_state=9
)
rng = np.random.default_rng(9)
missing = rng.random(X.shape) < 0.12
X[missing] = np.nan
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.30, stratify=y, random_state=9
)

imputed_tree = make_pipeline(
    SimpleImputer(strategy="median", add_indicator=True),
    DecisionTreeClassifier(max_depth=6, min_samples_leaf=12, random_state=9),
)
native_hist_tree = HistGradientBoostingClassifier(
    max_leaf_nodes=15, learning_rate=0.08, max_iter=150,
    early_stopping=True, random_state=9
)

for name, model in [("imputed single tree", imputed_tree),
                    ("native-NaN histogram boosting", native_hist_tree)]:
    model.fit(X_train, y_train)
    probability = model.predict_proba(X_test)[:, 1]
    print(f"{name:>31}: test ROC AUC={roc_auc_score(y_test, probability):.3f}")
```

</details>

<details>
<summary><strong>Python example: find a surrogate threshold that mimics a primary split</strong></summary>

```python
import numpy as np

rng = np.random.default_rng(4)
primary_feature = rng.normal(size=80)
correlated_feature = 0.85 * primary_feature + rng.normal(0, 0.45, size=80)
primary_left = primary_feature <= 0.0

candidates = (np.sort(np.unique(correlated_feature))[:-1]
              + np.sort(np.unique(correlated_feature))[1:]) / 2
best = None
for threshold in candidates:
    for direction in ["le", "gt"]:
        surrogate_left = (correlated_feature <= threshold)
        if direction == "gt":
            surrogate_left = ~surrogate_left
        agreement = np.mean(surrogate_left == primary_left)
        candidate = (agreement, threshold, direction)
        if best is None or candidate[0] > best[0]:
            best = candidate

agreement, threshold, direction = best
print(f"primary rule: x1 <= 0")
print(f"surrogate rule: x2 {direction} {threshold:.3f}")
print(f"routing agreement: {agreement:.3f}")
```

</details>

Split handling is part of the estimator, not a cosmetic preprocessing choice. Two tools both described as "decision trees" can learn different models from the same table because their categorical, missing, weighting, and stopping rules differ.


### **Tree Complexity and Pruning**

An unrestricted tree can continue until leaves are pure or too small to split. On noise-free training data, that may give zero classification error; on new data, the tree can be brittle because later splits are supported by only a few observations. Tree regularization controls the number and reliability of regions rather than shrinking coefficients.

A binary tree of depth $d$ can have up to $2^d$ leaves. Depth is therefore a coarse capacity measure: one extra level can double the number of terminal regions. Leaf count and minimum leaf support often describe effective complexity more directly than depth alone.

#### **Pre-Pruning**

**Pre-pruning** stops growth before the tree becomes maximal. Common controls include:

- `max_depth`: maximum number of decisions on a path;
- `max_leaf_nodes`: direct cap on the number of terminal regions;
- `min_samples_split`: support required before a node may be split;
- `min_samples_leaf`: support required in each resulting leaf;
- `min_impurity_decrease`: minimum weighted improvement needed for a split;
- `max_features`: number or fraction of features considered at a split.

`min_samples_leaf` is especially useful because it prevents extreme local estimates. In classification it reduces tiny, overconfident probability estimates; in regression it forces every step to average several targets. These values should be selected on development folds. Choosing depth from test performance converts the test set into training information.

<details>
<summary><strong>Python example: tune pre-pruning controls inside the development set</strong></summary>

```python
from sklearn.datasets import load_breast_cancer
from sklearn.metrics import roc_auc_score
from sklearn.model_selection import GridSearchCV, StratifiedKFold, train_test_split
from sklearn.tree import DecisionTreeClassifier

X, y = load_breast_cancer(return_X_y=True)
X_dev, X_test, y_dev, y_test = train_test_split(
    X, y, test_size=0.25, stratify=y, random_state=21
)
cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=21)
search = GridSearchCV(
    DecisionTreeClassifier(random_state=21),
    param_grid={
        "max_depth": [2, 3, 5, None],
        "min_samples_leaf": [1, 5, 15],
        "min_impurity_decrease": [0.0, 0.002],
    },
    scoring="roc_auc",
    cv=cv,
    n_jobs=1,
)
search.fit(X_dev, y_dev)

test_probability = search.best_estimator_.predict_proba(X_test)[:, 1]
print("best development settings:", search.best_params_)
print("development CV ROC AUC:", round(search.best_score_, 3))
print("untouched test ROC AUC:", round(roc_auc_score(y_test, test_probability), 3))
print("selected depth/leaves:", search.best_estimator_.get_depth(),
      search.best_estimator_.get_n_leaves())
```

</details>

#### **Minimal Cost-Complexity Post-Pruning**

**Post-pruning** first grows a large tree and then removes branches whose predictive improvement is too small for their complexity. CART's minimal cost-complexity objective is

$$
R_\alpha(T)=R(T)+\alpha|\widetilde T|,
$$

where $R(T)$ is the total leaf impurity or training risk, $|\widetilde T|$ is the number of leaves, and $\alpha\ge 0$ is the complexity price per leaf. At $\alpha=0$, fit dominates. As $\alpha$ grows, increasingly large subtrees must justify their extra leaves.

Weakest-link pruning produces a nested sequence of candidate subtrees. For an internal node $t$ with subtree $T_t$, its effective pruning value compares the risk increase from replacing the subtree by one leaf with the number of leaves removed:

$$
\alpha_{\mathrm{eff}}(t)=
\frac{R(t)-R(T_t)}{|\widetilde T_t|-1}.
$$

The smallest effective value is pruned first. Cross-validation then chooses among the resulting candidates. The path is data-dependent, so it must be computed from the development data within the experimental protocol.

![Tree node count and depth decrease stepwise as the cost-complexity parameter alpha increases.](assets/tree-pruning-complexity.png){fig-alt="Two step plots showing decision-tree node count and depth decreasing as cost-complexity alpha increases."}

*Official scikit-learn illustration: [cost-complexity pruning path](https://scikit-learn.org/stable/auto_examples/tree/plot_cost_complexity_pruning.html).*

<details>
<summary><strong>Python example: choose a cost-complexity subtree with cross-validation</strong></summary>

```python
import numpy as np
from sklearn.datasets import load_breast_cancer
from sklearn.metrics import accuracy_score
from sklearn.model_selection import StratifiedKFold, cross_val_score, train_test_split
from sklearn.tree import DecisionTreeClassifier

X, y = load_breast_cancer(return_X_y=True)
X_dev, X_test, y_dev, y_test = train_test_split(
    X, y, test_size=0.25, stratify=y, random_state=8
)

# Generate the nested pruning path using development data only.
path = DecisionTreeClassifier(random_state=8).cost_complexity_pruning_path(X_dev, y_dev)
nontrivial = path.ccp_alphas[:-1]
# Quantiles keep the teaching example fast while spanning the path.
alphas = np.unique(np.quantile(nontrivial, np.linspace(0, 1, 14)))
cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=8)

mean_scores = []
for alpha in alphas:
    candidate = DecisionTreeClassifier(ccp_alpha=float(alpha), random_state=8)
    mean_scores.append(cross_val_score(candidate, X_dev, y_dev, cv=cv).mean())

best_alpha = float(alphas[int(np.argmax(mean_scores))])
pruned = DecisionTreeClassifier(ccp_alpha=best_alpha, random_state=8).fit(X_dev, y_dev)
unpruned = DecisionTreeClassifier(random_state=8).fit(X_dev, y_dev)

for name, model in [("unpruned", unpruned), ("CV-pruned", pruned)]:
    print(f"{name:>9}: nodes={model.tree_.node_count:3d}, depth={model.get_depth():2d}, "
          f"test accuracy={accuracy_score(y_test, model.predict(X_test)):.3f}")
print("selected ccp_alpha:", round(best_alpha, 6))
```

</details>

#### **Instability and the Motivation for Ensembles**

A tree's first split affects every later candidate subset. If two early features offer nearly equal impurity reductions, a small sample perturbation can swap their order and produce a different topology. This is **high variance**: the learning procedure is sensitive to which observations happen to appear in the training set.

Instability is harmful for a single model but useful for aggregation. If many perturbed trees make errors that are not perfectly correlated, averaging can cancel part of their variation. Bagging and random forests formalize exactly this idea.

<details>
<summary><strong>Python example: observe root-split instability across bootstrap samples</strong></summary>

```python
from collections import Counter
import numpy as np
from sklearn.tree import DecisionTreeClassifier

rng = np.random.default_rng(5)
signal = rng.normal(size=420)
y = (signal + rng.normal(0, 0.8, size=420) > 0).astype(int)
# The first two columns are exact substitutes, so their best root splits tie.
X = np.column_stack([signal, signal, rng.normal(size=(420, 5))])
root_features = []
case_probabilities = []

for seed in range(30):
    indices = rng.integers(0, len(y), size=len(y))
    model = DecisionTreeClassifier(max_depth=4, random_state=seed)
    model.fit(X[indices], y[indices])
    root_features.append(int(model.tree_.feature[0]))
    case_probabilities.append(model.predict_proba(X[[0]])[0, 1])

print("root feature frequencies:", dict(sorted(Counter(root_features).items())))
print("one case's probability range:",
      tuple(np.round([min(case_probabilities), max(case_probabilities)], 3)))
print("probability standard deviation:", round(float(np.std(case_probabilities)), 3))
```

</details>

Pre-pruning is computationally direct; post-pruning compares nested subtrees after seeing a richer structure. Neither dominates universally. The selected tree should be judged by validation performance, stability, calibration, and the complexity a reader can realistically inspect.


### **Bagging and Random Forests**

Bagging and random forests do not try to make one tree stable. They construct many high-variance trees under controlled perturbations and average them. The essential requirements are that individual trees are reasonably accurate and that their errors are not perfectly correlated.

<div class="diagram-scroll">

![Random forests use bootstrap samples and random feature subsets to create diverse trees, aggregate their predictions, and estimate out-of-bag error.](assets/random-forest-mechanism.svg){fig-alt="Training rows are bootstrap sampled into multiple datasets, random-feature trees are fit in parallel, predictions are aggregated, and omitted cases form out-of-bag evaluations."}

</div>

#### **Bootstrap Aggregation**

For each $b=1,\ldots,B$, bagging samples $n$ training rows **with replacement**, fits a base learner $T_b$, and aggregates predictions. Regression averages,

$$
\hat f_{\mathrm{bag}}(x)=\frac{1}{B}\sum_{b=1}^{B}T_b(x),
$$

while classification commonly averages class probabilities and then takes the largest component. Averaging probabilities retains confidence information and gives a more stable tie policy than a raw class vote.

A particular training case is absent from one bootstrap sample with probability

$$
\left(1-\frac{1}{n}\right)^n\longrightarrow e^{-1}\approx 0.368.
$$

Thus a bootstrap sample of size $n$ contains about $63.2\%$ unique training cases in expectation; the rest of its positions are duplicates. The exact fraction varies from sample to sample.

Averaging only helps with reducible variation. If each tree prediction has variance $\sigma^2$ and pairwise correlation $\rho$, the variance of their mean is approximately

$$
\operatorname{Var}(\bar T)=\rho\sigma^2+\frac{1-\rho}{B}\sigma^2.
$$

Increasing $B$ shrinks the second term but cannot remove the correlated component $\rho\sigma^2$. This explains why simply adding more nearly identical trees eventually plateaus, and why a random forest also randomizes feature selection.

![Bias-variance decomposition for a single regression tree and a bagged ensemble of trees.](assets/bagging-bias-variance.png){fig-alt="A single regression tree has a wider spread of fitted curves and higher variance than a bagged ensemble, whose averaged predictions are more stable."}

*Official scikit-learn illustration: [single tree versus bagging bias-variance decomposition](https://scikit-learn.org/stable/auto_examples/ensemble/plot_bias_variance.html).*

<details>
<summary><strong>Python example: inspect bootstrap duplicates and out-of-bag observations</strong></summary>

```python
import numpy as np

rng = np.random.default_rng(17)
n = 20
bootstrap = rng.integers(0, n, size=n)
counts = np.bincount(bootstrap, minlength=n)
in_bag = np.flatnonzero(counts > 0)
out_of_bag = np.flatnonzero(counts == 0)

print("bootstrap row IDs:", bootstrap.tolist())
print("unique in-bag IDs:", in_bag.tolist())
print("out-of-bag IDs:", out_of_bag.tolist())
print("unique fraction:", round(len(in_bag) / n, 3))
print("largest duplicate count:", int(counts.max()))
```

</details>

<details>
<summary><strong>Python example: compare the sampling variance of one tree and bagged trees</strong></summary>

```python
import numpy as np
from sklearn.datasets import make_moons
from sklearn.ensemble import BaggingClassifier
from sklearn.metrics import accuracy_score
from sklearn.tree import DecisionTreeClassifier

# Use one large fixed test population and repeatedly draw small training sets.
X_test, y_test = make_moons(n_samples=4000, noise=0.30, random_state=999)
scores = {"single tree": [], "bagging": []}

for seed in range(20):
    X_train, y_train = make_moons(n_samples=160, noise=0.30, random_state=seed)
    tree = DecisionTreeClassifier(random_state=seed)
    bag = BaggingClassifier(
        estimator=DecisionTreeClassifier(), n_estimators=70,
        bootstrap=True, random_state=seed, n_jobs=1
    )
    for name, model in [("single tree", tree), ("bagging", bag)]:
        model.fit(X_train, y_train)
        scores[name].append(accuracy_score(y_test, model.predict(X_test)))

for name, values in scores.items():
    print(f"{name:>11}: mean accuracy={np.mean(values):.3f}, "
          f"between-sample SD={np.std(values):.3f}")
```

</details>

#### **Random Feature Subspaces**

Plain bagging gives every split access to all $p$ features. If one predictor is overwhelmingly strong, many bootstrap trees choose it near the root and remain correlated. A **random forest** samples a subset of candidate features at every node and selects the best split only within that subset. This can slightly weaken each tree while making the ensemble substantially more diverse.

`max_features` controls this tradeoff. A common classification default is approximately $\sqrt p$, but a default is not a theorem and differs across tasks and libraries. Too few candidate features can create high-bias trees; too many can leave the forest highly correlated. Tune it together with leaf size and, when needed, row subsampling.

**Extremely Randomized Trees** add more randomness by drawing candidate thresholds rather than optimizing every threshold exactly. They often train quickly and can reduce variance further, but the extra randomization can increase bias. The data regime determines whether the tradeoff helps.

| Method | Row randomization | Feature/threshold randomization | Main effect |
|---|---|---|---|
| Single tree | None | None | Low bias when deep, high variance |
| Bagging | Bootstrap rows | Usually all features, optimized thresholds | Reduces variance through row perturbation |
| Random forest | Usually bootstrap rows | Random feature subset, optimized thresholds | Further decorrelates trees |
| Extra Trees | Often full data by default, configurable | Random feature subset and randomized thresholds | More diversity and often faster split search |

<details>
<summary><strong>Python example: compare a tree, bagging, random forest, and Extra Trees</strong></summary>

```python
from sklearn.datasets import make_classification
from sklearn.ensemble import BaggingClassifier, ExtraTreesClassifier, RandomForestClassifier
from sklearn.metrics import accuracy_score
from sklearn.model_selection import train_test_split
from sklearn.tree import DecisionTreeClassifier

X, y = make_classification(
    n_samples=1800, n_features=30, n_informative=8, n_redundant=8,
    class_sep=0.85, flip_y=0.05, random_state=14
)
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.30, stratify=y, random_state=14
)

models = {
    "single tree": DecisionTreeClassifier(min_samples_leaf=3, random_state=14),
    "bagging": BaggingClassifier(
        estimator=DecisionTreeClassifier(min_samples_leaf=3),
        n_estimators=180, random_state=14, n_jobs=1
    ),
    "random forest": RandomForestClassifier(
        n_estimators=180, max_features="sqrt", min_samples_leaf=2,
        random_state=14, n_jobs=1
    ),
    "extra trees": ExtraTreesClassifier(
        n_estimators=180, max_features="sqrt", min_samples_leaf=2,
        random_state=14, n_jobs=1
    ),
}
for name, model in models.items():
    model.fit(X_train, y_train)
    print(f"{name:>13}: test accuracy={accuracy_score(y_test, model.predict(X_test)):.3f}")
```

</details>

#### **Out-of-Bag Evaluation**

Every training case is omitted by a subset of trees. Its **out-of-bag (OOB) prediction** aggregates only those trees, so the case was not used to fit any contributing tree. Averaging the case-level losses yields an internal estimate of generalization performance without a separate validation fit.

OOB evaluation is valuable for a quick learning curve or diagnostic, but it has boundaries:

- with very few trees, some cases have too few OOB predictions;
- repeated tuning against the OOB score can overfit that estimate;
- ordinary row bootstrapping is invalid for grouped, temporal, spatial, or otherwise dependent observations;
- the final untouched test set still serves a different purpose.

![Out-of-bag error trajectories for random forests with different feature-subset settings.](assets/random-forest-oob.png){fig-alt="OOB classification error decreases and stabilizes as the number of random-forest trees grows, with different curves for feature-subset sizes."}

*Official scikit-learn illustration: [random-forest OOB error trajectories](https://scikit-learn.org/stable/auto_examples/ensemble/plot_ensemble_oob.html).*

<details>
<summary><strong>Python example: compare OOB and held-out performance</strong></summary>

```python
from sklearn.datasets import load_breast_cancer
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import accuracy_score, roc_auc_score
from sklearn.model_selection import train_test_split

X, y = load_breast_cancer(return_X_y=True)
X_dev, X_test, y_dev, y_test = train_test_split(
    X, y, test_size=0.25, stratify=y, random_state=31
)
forest = RandomForestClassifier(
    n_estimators=400, max_features="sqrt", min_samples_leaf=2,
    bootstrap=True, oob_score=True, random_state=31, n_jobs=1
)
forest.fit(X_dev, y_dev)
test_probability = forest.predict_proba(X_test)[:, 1]

print("OOB accuracy estimate:", round(forest.oob_score_, 3))
print("held-out test accuracy:", round(accuracy_score(y_test, forest.predict(X_test)), 3))
print("held-out test ROC AUC:", round(roc_auc_score(y_test, test_probability), 3))
```

</details>

Random forests are strong low-maintenance tabular baselines: they model nonlinear interactions, tolerate unscaled numerical inputs, parallelize across trees, and usually require less delicate tuning than boosting. They can still be memory-heavy, produce rough probability estimates, struggle to extrapolate smooth trends, and obscure the concise explanation available from one small tree. Class weights, decision thresholds, and calibration must be handled explicitly in imbalanced or cost-sensitive tasks.


### **Boosting**

Boosting constructs an additive model from weak learners,

$$
F_M(x)=F_0(x)+\sum_{m=1}^{M}\eta\,\gamma_m h_m(x),
$$

where $h_m$ is commonly a shallow tree, $\gamma_m$ is its fitted contribution, and $\eta$ is a learning rate. Unlike bagging, these learners are not exchangeable: stage $m$ is trained using the current ensemble's errors or loss gradient. Training is therefore sequential.

<div class="diagram-scroll">

![Gradient boosting repeatedly computes the current loss signal, fits a weak tree, and updates the additive ensemble.](assets/boosting-sequence.svg){fig-alt="Gradient boosting initializes a constant model, computes pseudo-residuals, fits a shallow tree, updates the ensemble with a learning rate, and repeats."}

</div>

Boosting often reduces the bias of shallow trees and is exceptionally effective on structured tabular data. Its sequential correction also makes it sensitive to noisy labels, outliers, excessive tree complexity, and too many stages. Learning rate, tree size, row/feature subsampling, regularization, and early stopping determine whether later trees learn useful residual structure or chase noise.

#### **AdaBoost and Weak Learners**

Binary AdaBoost maintains a distribution of weights over training observations. Start with $w_i^{(1)}=1/n$. At round $m$, fit classifier $h_m(x)\in\{-1,+1\}$ to minimize weighted error

$$
\epsilon_m=\frac{\sum_i w_i^{(m)}\mathbf 1[y_i\ne h_m(x_i)]}{\sum_i w_i^{(m)}}.
$$

A learner better than chance receives weight

$$
\alpha_m=\frac{1}{2}\log\frac{1-\epsilon_m}{\epsilon_m},
$$

and observation weights are updated by

$$
w_i^{(m+1)}=\frac{w_i^{(m)}\exp[-\alpha_m y_i h_m(x_i)]}{Z_m}.
$$

Correct cases multiply by $e^{-\alpha_m}$; incorrect cases multiply by $e^{\alpha_m}$ and become more influential in the next round. $Z_m$ normalizes the weights. The final classifier is

$$
\hat y(x)=\operatorname{sign}\left(\sum_{m=1}^{M}\alpha_m h_m(x)\right).
$$

Decision stumps are common weak learners because each stage then adds one simple threshold. If $\epsilon_m=0.5$, a binary learner contributes no information; if it is worse than chance, its polarity can be reversed under the standard analysis. Near-zero error produces a very large weight, so implementations clip values and use learning-rate shrinkage for numerical and statistical stability.

<details>
<summary><strong>Python example: calculate one AdaBoost round from first principles</strong></summary>

```python
import numpy as np

x = np.array([-2.0, -1.3, -0.6, 0.1, 0.8, 1.5, 2.2])
y = np.array([-1,   -1,    1,   1,   1,  -1,  -1])
weights = np.full(len(y), 1 / len(y))
thresholds = (x[:-1] + x[1:]) / 2

best = None
for threshold in thresholds:
    for polarity in [1, -1]:
        prediction = np.where(x <= threshold, -1, 1)
        prediction *= polarity
        error = np.sum(weights[prediction != y])
        if best is None or error < best[0]:
            best = (error, threshold, polarity, prediction.copy())

error, threshold, polarity, prediction = best
alpha = 0.5 * np.log((1 - error) / error)
new_weights = weights * np.exp(-alpha * y * prediction)
new_weights /= new_weights.sum()

print(f"best stump: threshold={threshold:.2f}, polarity={polarity:+d}")
print(f"weighted error={error:.3f}, learner weight alpha={alpha:.3f}")
print("incorrect rows:", np.flatnonzero(prediction != y).tolist())
print("old weights:", np.round(weights, 3))
print("new weights:", np.round(new_weights, 3))
```

</details>

<details>
<summary><strong>Python example: compare one stump with an AdaBoost ensemble</strong></summary>

```python
from sklearn.datasets import make_moons
from sklearn.ensemble import AdaBoostClassifier
from sklearn.metrics import accuracy_score, roc_auc_score
from sklearn.model_selection import train_test_split
from sklearn.tree import DecisionTreeClassifier

X, y = make_moons(n_samples=1100, noise=0.28, random_state=18)
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.30, stratify=y, random_state=18
)
stump = DecisionTreeClassifier(max_depth=1, random_state=18)
boosted = AdaBoostClassifier(
    estimator=DecisionTreeClassifier(max_depth=1),
    n_estimators=140, learning_rate=0.45, random_state=18
)

for name, model in [("one stump", stump), ("AdaBoost", boosted)]:
    model.fit(X_train, y_train)
    probability = model.predict_proba(X_test)[:, 1]
    print(f"{name:>9}: accuracy={accuracy_score(y_test, model.predict(X_test)):.3f}, "
          f"ROC AUC={roc_auc_score(y_test, probability):.3f}")
```

</details>

#### **Gradient Boosting as Functional Optimization**

Gradient boosting generalizes the correction idea to a differentiable loss $L(y,F(x))$. It performs steepest descent in **function space** rather than updating a fixed coefficient vector.

1. Choose the best constant initialization:

$$
F_0=\arg\min_\gamma\sum_i L(y_i,\gamma).
$$

2. At stage $m$, calculate the negative gradient at every training point:

$$
r_{im}=-\left.\frac{\partial L(y_i,F(x_i))}{\partial F(x_i)}\right|_{F=F_{m-1}}.
$$

3. Fit a weak learner $h_m(x)$ to the pseudo-residuals $r_{im}$, optionally find a step size $\gamma_m$, and update

$$
F_m(x)=F_{m-1}(x)+\eta\gamma_mh_m(x).
$$

Under squared error $L=(y-F)^2/2$, the negative gradient is the familiar residual $y-F$. Under logistic loss, it is related to the difference between the observed label and current class probability. This is why the same boosting machinery can optimize regression, classification, ranking, robust, and quantile losses by changing the objective and derivatives.

<details>
<summary><strong>Python example: build squared-error gradient boosting from shallow regression trees</strong></summary>

```python
import numpy as np
from sklearn.metrics import mean_squared_error
from sklearn.tree import DecisionTreeRegressor

rng = np.random.default_rng(22)
X = np.linspace(-3, 3, 180).reshape(-1, 1)
y = 0.6 * X[:, 0] + 1.7 * np.sin(1.8 * X[:, 0]) + rng.normal(0, 0.25, len(X))

prediction = np.full(len(y), y.mean())  # F_0 minimizes squared error.
learning_rate = 0.18
print("stage 0 MSE:", round(mean_squared_error(y, prediction), 4))

learners = []
for stage in range(1, 7):
    negative_gradient = y - prediction
    tree = DecisionTreeRegressor(max_depth=2, min_samples_leaf=12, random_state=stage)
    tree.fit(X, negative_gradient)
    prediction += learning_rate * tree.predict(X)
    learners.append(tree)
    print(f"stage {stage} MSE: {mean_squared_error(y, prediction):.4f}")
```

</details>

The number of boosting stages and learning rate are coupled. A smaller $\eta$ makes each update conservative and usually needs a larger $M$. Shallow trees restrict interaction order; a stump models additive main effects, while deeper trees can represent higher-order interactions. `subsample < 1` creates stochastic gradient boosting, which can reduce correlation and variance at the cost of noisier gradients.

![Training and test deviance over gradient-boosting iterations.](assets/gradient-boosting-deviance.png){fig-alt="Training deviance continues downward while held-out deviance reaches a minimum and then changes across boosting iterations."}

*Official scikit-learn illustration: [gradient-boosting regression deviance](https://scikit-learn.org/stable/auto_examples/ensemble/plot_gradient_boosting_regression.html).*

![Test deviance trajectories under gradient-boosting shrinkage, row subsampling, and feature subsampling.](assets/gradient-boosting-regularization.png){fig-alt="Five test-deviance curves compare no shrinkage with learning-rate, row-subsampling, and feature-subsampling regularization."}

*Official scikit-learn illustration: [gradient-boosting regularization](https://scikit-learn.org/stable/auto_examples/ensemble/plot_gradient_boosting_regularization.html).*

<details>
<summary><strong>Python example: select the number of boosting stages without using the test set</strong></summary>

```python
import numpy as np
from sklearn.datasets import make_classification
from sklearn.ensemble import GradientBoostingClassifier
from sklearn.metrics import log_loss, roc_auc_score
from sklearn.model_selection import train_test_split

X, y = make_classification(
    n_samples=1800, n_features=18, n_informative=8, n_redundant=4,
    class_sep=0.8, flip_y=0.12, random_state=10
)
X_dev, X_test, y_dev, y_test = train_test_split(
    X, y, test_size=0.20, stratify=y, random_state=10
)
X_train, X_valid, y_train, y_valid = train_test_split(
    X_dev, y_dev, test_size=0.25, stratify=y_dev, random_state=10
)

candidate = GradientBoostingClassifier(
    n_estimators=400, learning_rate=0.08, max_depth=3,
    subsample=0.8, random_state=10
).fit(X_train, y_train)

validation_losses = [
    log_loss(y_valid, probability)
    for probability in candidate.staged_predict_proba(X_valid)
]
best_stage = int(np.argmin(validation_losses)) + 1

# Refit the selected configuration on all development observations.
final_model = GradientBoostingClassifier(
    n_estimators=best_stage, learning_rate=0.08, max_depth=3,
    subsample=0.8, random_state=10
).fit(X_dev, y_dev)
test_probability = final_model.predict_proba(X_test)[:, 1]

print("selected stage:", best_stage)
print("best validation log loss:", round(min(validation_losses), 4))
print("last candidate validation log loss:", round(validation_losses[-1], 4))
print("untouched test ROC AUC:", round(roc_auc_score(y_test, test_probability), 3))
```

</details>

#### **XGBoost, LightGBM, CatBoost, and Histogram Trees**

Modern gradient-boosting systems preserve the additive-tree principle but optimize how split candidates, derivatives, sparse values, categories, memory, and distributed computation are handled.

XGBoost popularized a regularized second-order approximation. For a new tree $f_t$, write

$$
\mathcal L^{(t)}\approx
\sum_i\left[g_i f_t(x_i)+\frac{1}{2}h_i f_t(x_i)^2\right]
+\Omega(f_t),
$$

where $g_i$ and $h_i$ are the first and second derivatives of the loss with respect to the current score. If a tree has $T$ leaves with values $w_j$ and

$$
\Omega(f)=\gamma T+\frac{\lambda}{2}\sum_{j=1}^{T}w_j^2,
$$

then for leaf $j$, with derivative sums $G_j=\sum_{i\in I_j}g_i$ and $H_j=\sum_{i\in I_j}h_i$, the regularized optimal value is

$$
w_j^*=-\frac{G_j}{H_j+\lambda}.
$$

Candidate split gain compares the regularized scores of the two children with the unsplit parent. This connects tree growth directly to loss reduction and explicit penalties.

| System | Characteristic design choices | Particularly useful when | Important caution |
|---|---|---|---|
| XGBoost | First/second derivatives, regularized objective, sparsity-aware and histogram/approximate split methods, row/column subsampling | A mature general-purpose boosted-tree stack is needed | Defaults and tree method vary by version/device; tune validation and early stopping |
| LightGBM | Histogram binning and best-first leaf-wise growth; supports native categorical partitioning and distributed training | Data are large and training throughput matters | Leaf-wise growth can overfit small data unless leaves, depth, and leaf support are constrained |
| CatBoost | Ordered target statistics, ordered boosting, and usually symmetric/oblivious trees | High-cardinality categorical features are central | Category semantics and train/inference preprocessing must remain consistent |
| Histogram GBDT | Bin continuous values before split search; often native missing and categorical support | A fast in-library baseline is desired for medium/large tabular data | Binning and native-feature capabilities differ across implementations |

CatBoost's "ordered" procedures are designed to prevent a row's target from directly determining its own category statistic, addressing a form of target leakage. LightGBM's leaf-wise strategy expands the leaf with the largest gain rather than growing every level uniformly. These descriptions are useful defaults, but each project evolves; always inspect the estimator version and its actual parameters.

<details>
<summary><strong>Python example: compare classical and histogram gradient boosting</strong></summary>

```python
import time
from sklearn.datasets import make_classification
from sklearn.ensemble import GradientBoostingClassifier, HistGradientBoostingClassifier
from sklearn.metrics import roc_auc_score
from sklearn.model_selection import train_test_split

X, y = make_classification(
    n_samples=5000, n_features=24, n_informative=10, n_redundant=6,
    class_sep=0.9, random_state=27
)
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.25, stratify=y, random_state=27
)
models = {
    "classical GBDT": GradientBoostingClassifier(
        n_estimators=120, learning_rate=0.06, max_depth=2, random_state=27
    ),
    "histogram GBDT": HistGradientBoostingClassifier(
        max_iter=120, learning_rate=0.06, max_leaf_nodes=15,
        early_stopping=True, random_state=27
    ),
}

for name, model in models.items():
    start = time.perf_counter()
    model.fit(X_train, y_train)
    elapsed = time.perf_counter() - start
    auc = roc_auc_score(y_test, model.predict_proba(X_test)[:, 1])
    print(f"{name:>14}: test ROC AUC={auc:.3f}, fit time={elapsed:.2f}s")
```

</details>

Boosting should be stopped and selected using a validation protocol appropriate to the data structure. A random internal validation fraction is unsuitable for time series or grouped records. Probability calibration and threshold tuning should use data not consumed by the boosting fit or early-stopping decision.


### **Voting and Stacking**

Tree ensembles combine variants of one model family. **Voting** and **stacking** can combine heterogeneous estimators whose inductive biases and error patterns differ. Diversity is more important than model count: averaging ten near-identical systems adds less information than combining a small set whose validated errors are complementary.

#### **Hard and Soft Voting**

Hard voting returns the most frequent predicted class,

$$
\hat y(x)=\operatorname{mode}\{h_1(x),\ldots,h_M(x)\}.
$$

Soft voting averages class probabilities, optionally with nonnegative weights $a_m$,

$$
\hat p_k(x)=\frac{\sum_{m=1}^{M}a_m\hat p_{mk}(x)}{\sum_m a_m},
\qquad
\hat y(x)=\arg\max_k\hat p_k(x).
$$

Soft voting uses more information but assumes the probability scales are comparable. An overconfident, poorly calibrated model can dominate the average even when its ranking performance is good. Calibrate base models on appropriate held-out data, and choose voting weights within the development protocol rather than from the final test set.

<details>
<summary><strong>Python example: compare base classifiers with soft voting</strong></summary>

```python
from sklearn.datasets import make_moons
from sklearn.ensemble import RandomForestClassifier, VotingClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score, log_loss
from sklearn.model_selection import train_test_split
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.svm import SVC

X, y = make_moons(n_samples=1400, noise=0.30, random_state=36)
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.30, stratify=y, random_state=36
)
logistic = make_pipeline(StandardScaler(), LogisticRegression(C=1.0))
forest = RandomForestClassifier(
    n_estimators=220, min_samples_leaf=4, random_state=36, n_jobs=1
)
svm = make_pipeline(StandardScaler(), SVC(C=2.0, gamma="scale", probability=True,
                                           random_state=36))
voting = VotingClassifier(
    estimators=[("logistic", logistic), ("forest", forest), ("svm", svm)],
    voting="soft", weights=[1, 2, 2]
)

for name, model in [("logistic", logistic), ("forest", forest),
                    ("RBF SVM", svm), ("soft voting", voting)]:
    model.fit(X_train, y_train)
    probability = model.predict_proba(X_test)
    print(f"{name:>11}: accuracy={accuracy_score(y_test, model.predict(X_test)):.3f}, "
          f"log loss={log_loss(y_test, probability):.3f}")
```

</details>

#### **Stacking with Out-of-Fold Predictions**

Stacking learns how to combine base predictions. Let $f_1,\ldots,f_M$ be base models and $g$ a meta-model. The level-two feature vector is

$$
z_i=\left(f_1^{(-k(i))}(x_i),\ldots,f_M^{(-k(i))}(x_i)\right),
$$

where $f_m^{(-k(i))}$ was fitted without the fold containing observation $i$. The meta-model learns $g(z_i)\approx y_i$ from these **out-of-fold (OOF)** predictions.

Training $g$ on base predictions for rows that those same base fits have already seen is leakage. A high-capacity base learner may produce nearly perfect in-sample predictions, teaching the meta-model an error pattern that disappears on new data. After the OOF matrix is complete, each base model is refitted on all development data for deployment; new base predictions are passed to the already learned meta-model.

<div class="diagram-scroll wide-diagram">

![Leakage-safe stacking creates out-of-fold base predictions for the meta-model and refits base models on all development data for deployment.](assets/stacking-oof-workflow.svg){fig-alt="Each base model predicts held-out folds to form an OOF matrix, a meta-model learns from that matrix, and full-data base fits are used at deployment."}

</div>

The outer evaluation of a stack still needs a separate split or outer cross-validation loop. All base tuning, OOF generation, and meta-model fitting must occur inside each outer training partition. Otherwise the stack can inherit subtle leakage even when its immediate level-two matrix is OOF.

<details>
<summary><strong>Python example: implement leakage-safe classification stacking</strong></summary>

```python
import numpy as np
from sklearn.base import clone
from sklearn.datasets import make_classification
from sklearn.ensemble import RandomForestClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score, roc_auc_score
from sklearn.model_selection import StratifiedKFold, cross_val_predict, train_test_split
from sklearn.neighbors import KNeighborsClassifier
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import StandardScaler

X, y = make_classification(
    n_samples=1800, n_features=16, n_informative=8, n_redundant=3,
    class_sep=0.8, flip_y=0.05, random_state=44
)
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.30, stratify=y, random_state=44
)
base_models = [
    make_pipeline(StandardScaler(), LogisticRegression(C=0.5)),
    RandomForestClassifier(n_estimators=180, min_samples_leaf=3,
                           random_state=44, n_jobs=1),
    make_pipeline(StandardScaler(), KNeighborsClassifier(n_neighbors=17)),
]
cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=44)

# Every value in Z_train comes from a model that did not see that row.
Z_train = np.column_stack([
    cross_val_predict(model, X_train, y_train, cv=cv,
                      method="predict_proba", n_jobs=1)[:, 1]
    for model in base_models
])
meta = LogisticRegression(C=1.0).fit(Z_train, y_train)

# Deployment base fits use all training rows and produce test-level features.
fitted_base = [clone(model).fit(X_train, y_train) for model in base_models]
Z_test = np.column_stack([
    model.predict_proba(X_test)[:, 1] for model in fitted_base
])
probability = meta.predict_proba(Z_test)[:, 1]

print("meta coefficients:", np.round(meta.coef_[0], 3))
print("OOF meta training accuracy:", round(accuracy_score(y_train, meta.predict(Z_train)), 3))
print("stack test accuracy:", round(accuracy_score(y_test, probability >= 0.5), 3))
print("stack test ROC AUC:", round(roc_auc_score(y_test, probability), 3))
```

</details>

| Ensemble | Base training relationship | Combination rule | Typical reason to use it |
|---|---|---|---|
| Bagging | Parallel, perturbed samples | Average or vote | Reduce variance of an unstable learner |
| Random forest | Parallel, perturbed rows and split features | Average class probabilities or regression values | Reduce tree correlation while retaining flexible interactions |
| Boosting | Sequential, each stage depends on current loss | Additive weighted sum | Correct bias and optimize a chosen loss |
| Voting | Independent heterogeneous fits | Fixed vote/probability weights | Obtain a simple robust blend |
| Stacking | Base fits plus OOF level-two construction | Learned meta-model | Exploit systematic complementary errors |


### **Interpretation, Complexity, and Model Choice**

A small decision tree is directly inspectable: follow one path, read the thresholds, inspect the training support and class distribution in the leaf, and check whether the conditions are operationally meaningful. An ensemble replaces one path with hundreds or thousands of paths. It is still analysable, but interpretation becomes a separate statistical task rather than a literal reading of the model.

#### **Feature Importance and Local Structure**

Mean decrease in impurity (MDI) assigns feature $j$ the normalized sum of weighted impurity reductions at nodes that split on it:

$$
\operatorname{MDI}(j)\propto
\sum_{t:v(t)=j}\frac{n_t}{n}\left[
Q(t)-\frac{n_L}{n_t}Q(t_L)-\frac{n_R}{n_t}Q(t_R)
\right].
$$

It is fast because it is accumulated during fitting. However, it can favor continuous or high-cardinality features with many candidate thresholds, and correlated predictors can divide or substitute for each other's importance. It describes how the fitted forest used features, not the causal effect of intervening on them.

**Permutation importance** breaks one feature's association with the target by shuffling it and measures the loss increase on held-out data. It is model-agnostic and evaluates predictive dependence in the selected dataset, but correlated substitutes can make each individual permutation appear unimportant. Grouped permutation, conditional methods, and domain-defined feature groups can be more meaningful when predictors are redundant.

![Mean decrease in impurity importances from a random forest, with variability across trees.](assets/forest-feature-importances.png){fig-alt="A bar chart of random-forest mean decrease in impurity feature importances with variability across trees."}

*Official scikit-learn illustration: [forest feature importances](https://scikit-learn.org/stable/auto_examples/ensemble/plot_forest_importances.html).*

Partial dependence plots average predictions while varying selected features; ICE curves retain one line per observation and can reveal heterogeneous interactions. Accumulated local effects can be more reliable when features are correlated. SHAP-based tree explainers distribute a prediction among features efficiently for many tree ensembles, but the resulting attribution still depends on the chosen background distribution and assumptions about feature dependence. None of these tools converts an observational predictor into a causal explanation.

<details>
<summary><strong>Python example: contrast impurity and held-out permutation importance</strong></summary>

```python
import numpy as np
from sklearn.ensemble import RandomForestClassifier
from sklearn.inspection import permutation_importance
from sklearn.model_selection import train_test_split

rng = np.random.default_rng(52)
n = 2600
signal_1 = rng.normal(size=n)
signal_2 = rng.integers(0, 2, size=n)
y = (signal_1 + 0.9 * signal_2 + rng.normal(0, 0.75, size=n) > 0.45).astype(int)
X = np.column_stack([
    signal_1,
    signal_2,
    rng.normal(size=n),          # high-cardinality continuous noise
    rng.integers(0, 4, size=n),  # low-cardinality noise
])
names = np.array(["signal_cont", "signal_binary", "noise_cont", "noise_4level"])
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.35, stratify=y, random_state=52
)
forest = RandomForestClassifier(n_estimators=350, random_state=52, n_jobs=1)
forest.fit(X_train, y_train)
permutation = permutation_importance(
    forest, X_test, y_test, n_repeats=15, random_state=52, n_jobs=1
)

for name, mdi, perm, sd in zip(
    names, forest.feature_importances_,
    permutation.importances_mean, permutation.importances_std
):
    print(f"{name:>13}: MDI={mdi:.3f}, permutation={perm:+.3f} +/- {sd:.3f}")
```

</details>

#### **Computational and Statistical Tradeoffs**

Exact costs depend on implementation, data sparsity, histogram binning, and tree balance, so big-O summaries are orientation rather than runtime promises.

| Model | Fitting characteristics | Prediction and memory | Statistical profile |
|---|---|---|---|
| One small tree | Greedy split search; often near $O(pn\log n)$ under efficient sorting assumptions | Roughly path depth per case; compact | Interpretable, nonlinear, high variance |
| Bagging / forest | Approximately $B$ tree fits; parallel across trees | $B$ paths and all stored nodes | Strong variance reduction; performance plateaus as correlated variance remains |
| Extra Trees | Random thresholds reduce split-search work | Similar ensemble prediction/memory | More randomization, sometimes higher bias |
| Classical GBDT | Sequential tree fits over current pseudo-residuals | Sum over all stages | Strong accuracy; learning rate and stages are coupled |
| Histogram GBDT | Up-front binning reduces candidate split work | Compact bins during training; tree ensemble at inference | Efficient on larger data; native missing/category support may help |
| Stacking | Cross-validated base fits plus refits and meta-model | All base predictions plus meta inference | Can exploit diverse errors; expensive and leakage-sensitive |

Scaling numerical features is generally unnecessary for ordinary trees because thresholds depend on order. This does not make preprocessing irrelevant: category encoding, missing values, sample weights, leakage, temporal/group boundaries, outliers in regression targets, and train-serving consistency still matter. Trees also extrapolate poorly: outside the training range, a case remains in an existing terminal region and receives an existing leaf value.

#### **Choosing a Tree or Ensemble Strategy**

A practical decision process is:

1. Fit a regularized single tree when an inspectable rule hierarchy is required. Report its support and stability, not only its diagram.
2. Use a random forest or Extra Trees as a strong, low-tuning nonlinear baseline. Increase `n_estimators` until validation/OOB performance and prediction variance stabilize; tune leaf support and feature subsampling.
3. Use gradient or histogram boosting when predictive performance on tabular data is central. Tune learning rate with stages, tree leaves/depth, subsampling, regularization, and early stopping.
4. Prefer native categorical methods only when their leakage controls and deployment semantics are understood. Otherwise use a cross-fitted preprocessing pipeline.
5. Add voting or stacking only when base-model errors are demonstrably complementary. Evaluate the entire combination with outer validation.
6. Check probability calibration, threshold costs, subgroup errors, latency, memory, and explanation stability before deployment.

<details>
<summary><strong>Python example: select among tree families using development cross-validation</strong></summary>

```python
import numpy as np
from sklearn.base import clone
from sklearn.datasets import load_breast_cancer
from sklearn.ensemble import (
    ExtraTreesClassifier,
    GradientBoostingClassifier,
    HistGradientBoostingClassifier,
    RandomForestClassifier,
)
from sklearn.metrics import roc_auc_score
from sklearn.model_selection import StratifiedKFold, cross_val_score, train_test_split
from sklearn.tree import DecisionTreeClassifier

X, y = load_breast_cancer(return_X_y=True)
X_dev, X_test, y_dev, y_test = train_test_split(
    X, y, test_size=0.25, stratify=y, random_state=61
)
candidates = {
    "pruned tree": DecisionTreeClassifier(
        max_depth=4, min_samples_leaf=8, random_state=61
    ),
    "random forest": RandomForestClassifier(
        n_estimators=260, min_samples_leaf=2, max_features="sqrt",
        random_state=61, n_jobs=1
    ),
    "extra trees": ExtraTreesClassifier(
        n_estimators=260, min_samples_leaf=2, max_features="sqrt",
        random_state=61, n_jobs=1
    ),
    "gradient boosting": GradientBoostingClassifier(
        n_estimators=180, learning_rate=0.04, max_depth=2, random_state=61
    ),
    "histogram boosting": HistGradientBoostingClassifier(
        max_iter=180, learning_rate=0.05, max_leaf_nodes=15,
        l2_regularization=0.5, early_stopping=True, random_state=61
    ),
}
cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=61)
development_scores = {}
for name, model in candidates.items():
    scores = cross_val_score(model, X_dev, y_dev, scoring="roc_auc", cv=cv, n_jobs=1)
    development_scores[name] = scores.mean()
    print(f"{name:>18}: development ROC AUC={scores.mean():.3f} +/- {scores.std():.3f}")

selected_name = max(development_scores, key=development_scores.get)
selected = clone(candidates[selected_name]).fit(X_dev, y_dev)
test_auc = roc_auc_score(y_test, selected.predict_proba(X_test)[:, 1])
print("selected from development data:", selected_name)
print("one-time test ROC AUC:", round(test_auc, 3))
```

</details>

| Requirement | Usually start with | Why |
|---|---|---|
| A short, auditable decision policy | Pruned tree or carefully validated rule list | Direct paths and explicit conditions |
| Reliable nonlinear tabular baseline | Random forest | Stable, parallel, and relatively forgiving |
| Maximum tabular predictive performance | Regularized histogram boosting, then compare specialized libraries | Efficient nonlinear interactions and loss-based training |
| Many informative categorical variables | CatBoost or validated native categorical boosting | Ordered category handling can avoid naive one-hot explosion |
| Very low latency or memory | Small tree, distilled model, or constrained boosting | Ensemble size directly affects inference cost |
| Complementary established models | Weighted voting before stacking | Voting is simpler; stacking is justified only by validated incremental value |

The central distinction from Chapter 08 is geometric. KNN and kernels reason through distances or similarities in a chosen representation. Trees repeatedly partition coordinates into local regions. A useful benchmark compares both families under the same leakage-safe evaluation pipeline, because neither inductive bias is universally superior.
